# Module D: Financial Performance Intelligence & Scenario Modeling
## RISK Enterprise Decision Intelligence Platform
**Role**: Financial Intelligence Analyst & Modeler  
**Dataset**: Financial Sample Workbook (`data/raw/financial_sample.xlsx`)  
**Objective**: Audited financial statement analysis, segment/country margin distributions, discount leakage diagnostics, and parameter sensitivity simulation.


### 1. Ingestion & Accounting Identity Reconciliation
Loading the workbook, stripping header whitespace, and verifying that:
1. `Gross Sales - Discounts == Net Sales`
2. `Net Sales - COGS == Profit`


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '..')

from src.loaders import load_financial_sample
from src.cleaning import clean_financial_sample
from src.metrics import calculate_financial_kpis, simulate_financial_scenario
from src.visualization import set_reporting_style, PALETTE

set_reporting_style()

fin_raw = load_financial_sample('../data/raw/financial_sample.xlsx')
df = clean_financial_sample(fin_raw)
print(f"Audited Financial Records: {len(df):,} transactions across {df.shape[1]} metrics.")


### 2. Audited Executive Income Statement Scorecard
Comprehensive summary of total revenue, costs, and profit margin.


In [ ]:
kpis = calculate_financial_kpis(df)
for k, v in kpis.items():
    if isinstance(v, float):
        print(f"{k:<30}: ${v:,.2f}" if 'ratio' not in k and 'pct' not in k and 'unit' not in k else f"{k:<30}: {v:,.2f}%")
    else:
        print(f"{k:<30}: {v:,}")


### 3. Business Segment Financial Contribution
Analyzing Net Sales, Gross Profit, and COGS ratios by customer segment.


In [ ]:
seg_summary = df.groupby('Segment', observed=False).agg(
    Net_Sales=('Net Sales', 'sum'),
    Gross_Profit=('Profit', 'sum'),
    Total_COGS=('COGS', 'sum'),
    Units_Sold=('Units Sold', 'sum')
).reset_index()
seg_summary['Gross_Margin_Pct'] = (seg_summary['Gross_Profit'] / seg_summary['Net_Sales']) * 100
seg_summary['COGS_Ratio_Pct'] = (seg_summary['Total_COGS'] / seg_summary['Net_Sales']) * 100
display(seg_summary.sort_values('Net_Sales', ascending=False))


### 4. Geographic Margin Distribution
Evaluating profitability and market efficiency across Canada, Germany, France, Mexico, and USA.


In [ ]:
country_summary = df.groupby('Country', observed=False).agg(
    Net_Sales=('Net Sales', 'sum'),
    Gross_Profit=('Profit', 'sum')
).reset_index()
country_summary['Gross_Margin_Pct'] = (country_summary['Gross_Profit'] / country_summary['Net_Sales']) * 100
display(country_summary.sort_values('Gross_Margin_Pct', ascending=False))


### 5. Interactive Scenario Simulation Engine
*Disclaimer*: Scenarios represent mathematical stress-tests, not guaranteed financial forecasts.
Simulating 3 distinct parameter environments:
- **Scenario 1**: 10% Discount Rationalization + 5% Cost Reduction
- **Scenario 2**: 20% Discount Discipline with -2% Volume Elasticity
- **Scenario 3**: 5% Cost of Goods Sold Inflation Stress Test


In [ ]:
s1 = simulate_financial_scenario(df, discount_reduction_pct=0.10, cogs_reduction_pct=0.05, volume_change_pct=0.0)
s2 = simulate_financial_scenario(df, discount_reduction_pct=0.20, cogs_reduction_pct=0.0, volume_change_pct=-0.02)
s3 = simulate_financial_scenario(df, discount_reduction_pct=0.0, cogs_reduction_pct=-0.05, volume_change_pct=0.0)

print(f"Baseline Gross Profit:   ${kpis['total_profit']:,.2f} ({kpis['gross_profit_margin_pct']:.2f}% Margin)")
print(f"Scenario 1 Profit:       ${s1['simulated']['total_profit']:,.2f} (Delta: +${s1['impact']['profit_delta']:,.2f})")
print(f"Scenario 2 Profit:       ${s2['simulated']['total_profit']:,.2f} (Delta: +${s2['impact']['profit_delta']:,.2f})")
print(f"Scenario 3 Profit:       ${s3['simulated']['total_profit']:,.2f} (Delta: ${s3['impact']['profit_delta']:,.2f})")
